In [1]:
import re
import csv
import copy
import math
import json
import traceback
import numpy as np
import pandas as pd
from pathlib import Path
import numpy.linalg as npl
from itertools import product
from scipy.optimize import root
import matplotlib.pyplot as plt
from __future__ import annotations
from scipy.sparse.linalg import eigs
from matplotlib.patches import Circle, Patch
from scipy.sparse import csr_matrix, diags, eye
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from dataclasses import dataclass, replace, asdict, is_dataclass
from typing import Any, Dict, Iterable, Iterator, List, Mapping, Optional, Sequence, Tuple

In [2]:
# ============================================================
#  Parameters
# ============================================================

@dataclass(frozen=True)
class AreaParams:
    """
    Parameters for one brain area.

    N      : number of neurons in this area
    s      : local sparsity, defined by C_i / N_i
    f_exc  : fraction of excitatory neurons in this area
    J_exc  : local excitatory synaptic weight in this area
    g_inh  : local inhibitory strength ratio
    """
    N: int
    s: float
    f_exc: float
    J_exc: float
    g_inh: float


@dataclass(frozen=True)
class MultiAreaParams:
    """
    Parameters for the full multi-area network.

    s_cross[src, tgt] : cross-area sparsity from src to tgt
    J_cross[src, tgt] : cross-area excitatory weight from src to tgt

    activation: "tanh" or "threshold_linear"
    phi_max   : only used if activation == "threshold_linear"
    gamma     : threshold offset in phi(x) = max(0, x + gamma)
    """
    areas: Tuple[AreaParams, ...]
    s_cross: np.ndarray
    J_cross: np.ndarray
    allow_autapses: bool
    activation: str
    phi_max: float
    gamma: float
    dt: float
    T: float
    seed: int = 0


# ============================================================
#  Activation
# ============================================================

def phi(x: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    x = np.asarray(x, dtype=float)

    if p.activation == "tanh":
        return np.tanh(x) + 1.0

    if p.activation == "threshold_linear":
        y = np.maximum(0.0, x + p.gamma)
        if np.isfinite(p.phi_max):
            y = np.minimum(p.phi_max, y)
        return y

    raise ValueError(f"Unknown activation = {p.activation}")


def phi_prime(x: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    x = np.asarray(x, dtype=float)

    if p.activation == "tanh":
        y = np.tanh(x)
        return 1.0 - y * y

    if p.activation == "threshold_linear":
        g = ((x + p.gamma) >= 0.0).astype(float)
        if np.isfinite(p.phi_max):
            g = g * ((x + p.gamma) < p.phi_max)
        return g

    raise ValueError(f"Unknown activation = {p.activation}")


# ============================================================
#  Population bookkeeping
# ============================================================

def population_info(p: MultiAreaParams) -> Dict[str, Any]:
    pops = []
    pop_sizes = []
    pop_slices = {}
    area_slices = {}
    area_stats = []

    start = 0
    for area_id, ap in enumerate(p.areas):
        N_E = int(round(ap.f_exc * ap.N))
        N_I = ap.N - N_E

        if N_E < 0 or N_I < 0:
            raise ValueError(f"Area {area_id}: invalid E/I split")

        C = int(round(ap.s * ap.N))
        C_E = int(round(ap.f_exc * C))
        C_I = C - C_E

        e_slice = slice(start, start + N_E)
        pops.append((area_id, "E"))
        pop_slices[(area_id, "E")] = e_slice
        pop_sizes.append(N_E)
        start += N_E

        i_slice = slice(start, start + N_I)
        pops.append((area_id, "I"))
        pop_slices[(area_id, "I")] = i_slice
        pop_sizes.append(N_I)
        start += N_I

        area_slices[area_id] = slice(e_slice.start, i_slice.stop)

        area_stats.append(
            {
                "area_id": area_id,
                "N": ap.N,
                "N_E": N_E,
                "N_I": N_I,
                "C": C,
                "C_E": C_E,
                "C_I": C_I,
                "J_E": ap.J_exc,
                "J_I": -ap.g_inh * ap.J_exc,
            }
        )

    return {
        "pops": pops,
        "pop_sizes": np.asarray(pop_sizes, dtype=int),
        "N_total": start,
        "pop_slices": pop_slices,
        "area_slices": area_slices,
        "area_stats": area_stats,
    }


def _validate_params(p: MultiAreaParams) -> None:
    A = len(p.areas)

    if p.s_cross.shape != (A, A):
        raise ValueError(f"s_cross must have shape ({A}, {A})")
    if p.J_cross.shape != (A, A):
        raise ValueError(f"J_cross must have shape ({A}, {A})")

    for area_id, ap in enumerate(p.areas):
        if ap.N <= 0:
            raise ValueError(f"Area {area_id}: N must be positive")
        if not (0.0 <= ap.s <= 1.0):
            raise ValueError(f"Area {area_id}: s must be in [0, 1]")
        if not (0.0 <= ap.f_exc <= 1.0):
            raise ValueError(f"Area {area_id}: f_exc must be in [0, 1]")

    if np.any(p.s_cross < 0.0) or np.any(p.s_cross > 1.0):
        raise ValueError("All s_cross entries must be in [0, 1]")


def _candidate_pool(indices: np.ndarray, target_idx: int, allow_autapses: bool) -> np.ndarray:
    if allow_autapses or len(indices) == 0:
        return indices

    if indices[0] <= target_idx <= indices[-1]:
        return indices[indices != target_idx]
    return indices

In [3]:
# ============================================================
#  Connectivity
# ============================================================

def build_multi_area_connectivity(p: MultiAreaParams) -> Tuple[csr_matrix, Dict[str, Any]]:
    """
    Build the full sparse recurrent matrix W.
    Convention: W[i, j] = connection from neuron j to neuron i.
    """
    _validate_params(p)
    rng = np.random.default_rng(p.seed)
    info = population_info(p)

    A = len(p.areas)
    N_total = info["N_total"]
    pop_slices = info["pop_slices"]
    area_stats = info["area_stats"]

    rows: List[int] = []
    cols: List[int] = []
    data: List[float] = []

    for tgt_area in range(A):
        stat_t = area_stats[tgt_area]
        tgt_all = np.arange(info["area_slices"][tgt_area].start, info["area_slices"][tgt_area].stop)
        tgt_exc = np.arange(pop_slices[(tgt_area, "E")].start, pop_slices[(tgt_area, "E")].stop)
        tgt_inh = np.arange(pop_slices[(tgt_area, "I")].start, pop_slices[(tgt_area, "I")].stop)

        C_E_local = stat_t["C_E"]
        C_I_local = stat_t["C_I"]
        w_E_local = stat_t["J_E"]
        w_I_local = stat_t["J_I"]

        for i_tgt in tgt_all:
            exc_pool = _candidate_pool(tgt_exc, i_tgt, p.allow_autapses)
            inh_pool = _candidate_pool(tgt_inh, i_tgt, p.allow_autapses)

            if C_E_local > len(exc_pool):
                raise ValueError(
                    f"Area {tgt_area}: local C_E={C_E_local} exceeds available local excitatory pool={len(exc_pool)}"
                )
            if C_I_local > len(inh_pool):
                raise ValueError(
                    f"Area {tgt_area}: local C_I={C_I_local} exceeds available local inhibitory pool={len(inh_pool)}"
                )

            if C_E_local > 0:
                pre_exc = rng.choice(exc_pool, size=C_E_local, replace=False)
                rows.extend([i_tgt] * C_E_local)
                cols.extend(pre_exc.tolist())
                data.extend([w_E_local] * C_E_local)

            if C_I_local > 0:
                pre_inh = rng.choice(inh_pool, size=C_I_local, replace=False)
                rows.extend([i_tgt] * C_I_local)
                cols.extend(pre_inh.tolist())
                data.extend([w_I_local] * C_I_local)

            for src_area in range(A):
                if src_area == tgt_area:
                    continue

                src_exc = np.arange(pop_slices[(src_area, "E")].start, pop_slices[(src_area, "E")].stop)
                if len(src_exc) == 0:
                    continue

                s_ij = float(p.s_cross[src_area, tgt_area])
                if s_ij <= 0.0:
                    continue

                C_cross = int(round(s_ij * len(src_exc)))
                if C_cross == 0:
                    continue
                if C_cross > len(src_exc):
                    raise ValueError(
                        f"Cross ({src_area}->{tgt_area}): C_cross={C_cross} exceeds source excitatory pool={len(src_exc)}"
                    )

                pre_cross = rng.choice(src_exc, size=C_cross, replace=False)
                w_cross = float(p.J_cross[src_area, tgt_area])

                rows.extend([i_tgt] * C_cross)
                cols.extend(pre_cross.tolist())
                data.extend([w_cross] * C_cross)

    W = csr_matrix(
        (
            np.asarray(data, dtype=float),
            (np.asarray(rows, dtype=int), np.asarray(cols, dtype=int)),
        ),
        shape=(N_total, N_total),
    )

    meta = {
        "N_total": N_total,
        "pops": info["pops"],
        "pop_sizes": info["pop_sizes"],
        "pop_slices": pop_slices,
        "area_slices": info["area_slices"],
        "area_stats": area_stats,
        "nnz": W.nnz,
    }
    return W, meta


def lift_population_values_to_neurons(values: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    info = population_info(p)
    pops = info["pops"]
    pop_slices = info["pop_slices"]
    N_total = info["N_total"]

    out = np.empty(N_total, dtype=float)
    for k, (area_id, cell_type) in enumerate(pops):
        out[pop_slices[(area_id, cell_type)]] = values[k]
    return out

In [4]:
# ============================================================
#  Fixed point
# ============================================================

def solve_full_fixed_point(
    W: csr_matrix,
    p: MultiAreaParams,
    x0_init: Optional[np.ndarray] = None,
    dt_fp: Optional[float] = None,
    tol: float = 1e-6,
    max_iter: int = 200_000,
    verbose: bool = False,
    max_abs_x: float = 1e6,
    adaptive: bool = True,
) -> np.ndarray:
    """
    Solve dx/dt = -x + W phi(x) until the drift is small.
    """
    N = W.shape[0]
    if x0_init is None:
        x = np.zeros(N, dtype=float)
    else:
        x = np.asarray(x0_init, dtype=float).copy()
        if x.shape != (N,):
            raise ValueError(f"x0_init must have shape ({N},)")

    if dt_fp is None:
        dt_fp = p.dt
    if dt_fp <= 0.0:
        raise ValueError("dt_fp must be positive")

    prev_residual = np.inf

    for k in range(max_iter):
        r = phi(x, p)
        F = -x + W @ r
        residual = float(np.max(np.abs(F)))

        if verbose and (k < 10 or k % 1000 == 0):
            print(
                f"[iter {k:6d}] dt_fp={dt_fp:.3e}, residual={residual:.3e}, max|x|={np.max(np.abs(x)):.3e}"
            )

        if not np.all(np.isfinite(F)) or not np.all(np.isfinite(x)):
            if adaptive and dt_fp > 1e-8:
                dt_fp *= 0.5
                x = np.zeros(N, dtype=float) if x0_init is None else np.asarray(x0_init, dtype=float).copy()
                prev_residual = np.inf
                continue
            raise RuntimeError("Non-finite values encountered during fixed-point solve.")

        if residual < tol:
            return x

        x_new = x + dt_fp * F

        if not np.all(np.isfinite(x_new)) or float(np.max(np.abs(x_new))) > max_abs_x:
            if adaptive and dt_fp > 1e-8:
                dt_fp *= 0.5
                continue
            raise RuntimeError(f"State diverged during fixed-point solve. max|x_new|={np.max(np.abs(x_new)):.3e}")

        if adaptive and residual > prev_residual * 1.2 and dt_fp > 1e-8:
            dt_fp *= 0.5
            continue

        x = x_new
        prev_residual = residual

    raise RuntimeError(
        f"Fixed-point solver did not converge in {max_iter} iterations. residual={residual:.3e}, max|x|={np.max(np.abs(x)):.3e}"
    )

# ============================================================
#  Full stability matrix
# ============================================================

def build_full_stability_matrix(
    W: csr_matrix,
    p: MultiAreaParams,
    x_star: np.ndarray,
) -> csr_matrix:
    """
        A = -I + W @ diag(phi'(x_star))
    """
    N = W.shape[0]
    x_star = np.asarray(x_star, dtype=float)

    if x_star.shape != (N,):
        raise ValueError(f"x_star must have shape ({N},)")

    gain = phi_prime(x_star, p)  # 每个 presynaptic neuron 的增益
    D = diags(gain, offsets=0, shape=(N, N), format="csr")

    A = -eye(N, format="csr") + W @ D
    return A

# ============================================================
#  Full spectra
# ============================================================

def compute_connectivity_spectrum(W: csr_matrix) -> np.ndarray:

    W_dense = W.toarray()
    eigvals = np.linalg.eigvals(W_dense)
    return eigvals


def compute_linearized_operator_spectrum(
    W: csr_matrix,
    p: MultiAreaParams,
    x_star: np.ndarray,
) -> np.ndarray:

    N = W.shape[0]
    gain = phi_prime(x_star, p)

    M = (W @ diags(gain, 0, shape=(N, N), format="csr")).toarray()
    eigvals = np.linalg.eigvals(M)
    return eigvals


def compute_stability_spectrum(
    W: csr_matrix,
    p: MultiAreaParams,
    x_star: np.ndarray,
) -> np.ndarray:

    A = build_full_stability_matrix(W, p, x_star).toarray()
    eigvals = np.linalg.eigvals(A)
    return eigvals

In [5]:
# ============================================================
#  Simulation
# ============================================================
def simulate_dynamics(
    W: csr_matrix,
    p: MultiAreaParams,
    x_init: np.ndarray,
    T: Optional[float] = None,
    store_every: int = 1,
) -> Dict[str, np.ndarray]:
    x = np.asarray(x_init, dtype=float).copy()
    T_total = p.T if T is None else float(T)
    n_steps = int(round(T_total / p.dt))

    times = []
    traj = []

    for k in range(n_steps + 1):
        if k % store_every == 0:
            times.append(k * p.dt)
            traj.append(x.copy())

        if k == n_steps:
            break

        dx = -x + W @ phi(x, p)
        x = x + p.dt * dx

    return {"t": np.asarray(times, dtype=float), "x": np.asarray(traj, dtype=float)}


# ============================================================
#  Perturbations
# ============================================================

def constant_perturbation(
    p: MultiAreaParams,
    amplitude: float,
    area_weights: Optional[Sequence[float]] = None,
) -> np.ndarray:
    """
    Piecewise-constant perturbation over areas.

    delta_i = amplitude * area_weights[area_id]

    If area_weights is None, all areas use weight 1.0.
    """
    info = population_info(p)
    n_areas = len(p.areas)

    if area_weights is None:
        area_weights = [1.0] * n_areas

    if len(area_weights) != n_areas:
        raise ValueError(
            f"area_weights must have length {n_areas}, got {len(area_weights)}"
        )

    delta = np.zeros(info["N_total"], dtype=float)
    for area_id, weight in enumerate(area_weights):
        sl = info["area_slices"][area_id]
        delta[sl] = float(amplitude) * float(weight)

    return delta


def gaussian_perturbation(
    p: MultiAreaParams,
    sigma: float,
    rng: Optional[np.random.Generator] = None,
    zero_mean: bool = True,
    area_weights: Optional[Sequence[float]] = None,
    exact_l2_match_to: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Gaussian random perturbation with optional per-area scaling.

    Construction order:
      1. sample N(0, sigma) over all neurons
      2. (optional) subtract the within-area mean for each area
      3. multiply each area slice by area_weights[a]
      4. (optional) rescale each area slice so its L2 norm matches the
         corresponding area slice of exact_l2_match_to (per-area match)

    Parameters
    ----------
    sigma
        Standard deviation before any per-area / L2 rescaling.
    zero_mean
        If True, subtract the within-area sample mean from each area's
        slice of the perturbation (per-area zero-mean).
    area_weights
        Per-area multiplicative weight. None means [1.0] * n_areas.
        Setting area_weights[a] = 0 gives delta[area_slice_a] = 0 exactly.
    exact_l2_match_to
        If provided, rescale each area's slice of the perturbation so its
        per-area L2 norm equals the per-area L2 norm of this reference
        vector. Areas where the reference has zero norm are zeroed in the
        output as well.
    """
    info = population_info(p)
    n_areas = len(p.areas)

    if rng is None:
        rng = np.random.default_rng()

    sigma = float(sigma)
    if sigma <= 0:
        raise ValueError(f"sigma must be positive, got {sigma}")

    if area_weights is None:
        area_weights = [1.0] * n_areas
    if len(area_weights) != n_areas:
        raise ValueError(
            f"area_weights must have length {n_areas}, got {len(area_weights)}"
        )

    delta = rng.normal(loc=0.0, scale=sigma, size=info["N_total"]).astype(float)

    for area_id in range(n_areas):
        sl = info["area_slices"][area_id]
        if zero_mean:
            delta[sl] = delta[sl] - np.mean(delta[sl])
        delta[sl] = delta[sl] * float(area_weights[area_id])

    if exact_l2_match_to is not None:
        ref = np.asarray(exact_l2_match_to, dtype=float)
        for area_id in range(n_areas):
            sl = info["area_slices"][area_id]
            target = float(np.linalg.norm(ref[sl]))
            current = float(np.linalg.norm(delta[sl]))
            if current > 0.0:
                delta[sl] = delta[sl] * (target / current)
            else:
                delta[sl] = 0.0

    return delta

In [6]:
# ============================================================
# utilities
# ============================================================

def sanitize_name(name: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")
    return s if s else "case"


def to_serializable(obj):
    if is_dataclass(obj):
        return to_serializable(asdict(obj))
    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, complex):
        return {"real": obj.real, "imag": obj.imag}
    return obj


# ============================================================
# statistics
# ============================================================

def init_stats_buffer(n_areas: int):
    return {
        "t": [],
        "E_x_total": [],
        "E_phi_total": [],
        "E_x_by_area": [],
        "E_phi_by_area": [],
    }


def append_state_statistics(
    buffer: dict,
    t: float,
    x: np.ndarray,
    x_ref: np.ndarray,
    p,
    info: dict,
    phi_ref: np.ndarray,
):
    x = np.asarray(x, dtype=float)
    dx = x - x_ref

    phix = np.asarray(phi(x, p), dtype=float)
    dphi = phix - phi_ref

    buffer["t"].append(float(t))
    buffer["E_x_total"].append(float(np.mean(dx ** 2)))
    buffer["E_phi_total"].append(float(np.mean(dphi ** 2)))

    n_areas = len(p.areas)
    ex_area = np.empty(n_areas, dtype=float)
    ephi_area = np.empty(n_areas, dtype=float)

    for area_id in range(n_areas):
        sl = info["area_slices"][area_id]
        ex_area[area_id] = float(np.mean(dx[sl] ** 2))
        ephi_area[area_id] = float(np.mean(dphi[sl] ** 2))

    buffer["E_x_by_area"].append(ex_area)
    buffer["E_phi_by_area"].append(ephi_area)


def finalize_stats_buffer(buffer: dict):
    n_rows = len(buffer["t"])

    if n_rows == 0:
        return {
            "t": np.empty(0, dtype=float),
            "E_x_total": np.empty(0, dtype=float),
            "E_phi_total": np.empty(0, dtype=float),
            "E_x_by_area": np.empty((0, 0), dtype=float),
            "E_phi_by_area": np.empty((0, 0), dtype=float),
        }

    return {
        "t": np.asarray(buffer["t"], dtype=float),
        "E_x_total": np.asarray(buffer["E_x_total"], dtype=float),
        "E_phi_total": np.asarray(buffer["E_phi_total"], dtype=float),
        "E_x_by_area": np.vstack(buffer["E_x_by_area"]).astype(float),
        "E_phi_by_area": np.vstack(buffer["E_phi_by_area"]).astype(float),
    }


def statistics_from_trajectory(
    times: np.ndarray,
    traj: np.ndarray,
    x_ref: np.ndarray,
    p,
):
    info = population_info(p)
    phi_ref = np.asarray(phi(x_ref, p), dtype=float)

    buffer = init_stats_buffer(len(p.areas))
    for t, x in zip(times, traj):
        append_state_statistics(
            buffer=buffer,
            t=float(t),
            x=np.asarray(x, dtype=float),
            x_ref=x_ref,
            p=p,
            info=info,
            phi_ref=phi_ref,
        )

    return finalize_stats_buffer(buffer)


def merge_pre_post_statistics(
    pre_stats: dict,
    post_stats: dict,
    max_re_lambda_stability: float,
):
    # 避免 pre 的 t=0 和 post 的 t=0 重复保存
    post_start = 0
    if (
        pre_stats["t"].size > 0
        and post_stats["t"].size > 0
        and np.isclose(pre_stats["t"][-1], 0.0)
        and np.isclose(post_stats["t"][0], 0.0)
    ):
        post_start = 1

    merged_t = np.concatenate([
        pre_stats["t"],
        post_stats["t"][post_start:],
    ])

    merged_E_x_total = np.concatenate([
        pre_stats["E_x_total"],
        post_stats["E_x_total"][post_start:],
    ])

    merged_E_phi_total = np.concatenate([
        pre_stats["E_phi_total"],
        post_stats["E_phi_total"][post_start:],
    ])

    merged_E_x_by_area = np.concatenate([
        pre_stats["E_x_by_area"],
        post_stats["E_x_by_area"][post_start:],
    ], axis=0)

    merged_E_phi_by_area = np.concatenate([
        pre_stats["E_phi_by_area"],
        post_stats["E_phi_by_area"][post_start:],
    ], axis=0)

    is_post = np.concatenate([
        np.zeros(pre_stats["t"].shape[0], dtype=int),
        np.ones(post_stats["t"][post_start:].shape[0], dtype=int),
    ])

    max_re_arr = np.full(
        shape=merged_t.shape,
        fill_value=float(max_re_lambda_stability),
        dtype=float,
    )

    return {
        "t": merged_t,
        "is_post_perturbation": is_post,
        "E_x_total": merged_E_x_total,
        "E_phi_total": merged_E_phi_total,
        "E_x_by_area": merged_E_x_by_area,
        "E_phi_by_area": merged_E_phi_by_area,
        "max_re_lambda_stability": max_re_arr,
    }


# ============================================================
# pre-perturbation relaxation + hold
# ============================================================

def relax_to_fixed_point_and_collect_statistics(
    W: csr_matrix,
    p,
    x_init: np.ndarray,
    x_ref: np.ndarray,
    relax_E_tol: float,
    relax_dx_tol: float,
    max_relax_steps: int,
    hold_time: float,
    store_every: int,
):
    info = population_info(p)
    phi_ref = np.asarray(phi(x_ref, p), dtype=float)

    x = np.asarray(x_init, dtype=float).copy()
    t_now = 0.0

    buffer = init_stats_buffer(len(p.areas))
    append_state_statistics(
        buffer=buffer,
        t=t_now,
        x=x,
        x_ref=x_ref,
        p=p,
        info=info,
        phi_ref=phi_ref,
    )

    relax_step_reached = None
    relax_time_reached = None
    
    n_consecutive_ok = 5
    ok_count = 0
    
    for k in range(max_relax_steps):
        dx = -x + W @ phi(x, p)
        x = x + p.dt * dx
        t_now += p.dt
    
        # 用你真正关心的统计量来判定是否“足够接近终点”
        E_x_now = np.mean((x - x_ref) ** 2)
    
        if (k + 1) % store_every == 0:
            append_state_statistics(
                buffer=buffer,
                t=t_now,
                x=x,
                x_ref=x_ref,
                p=p,
                info=info,
                phi_ref=phi_ref,
            )
    
        # # 方案 A：只看 E_x
        # if E_x_now < relax_E_tol:
        #     ok_count += 1
        # else:
        #     ok_count = 0
    
        #方案 B：如果你想更稳一点，可以同时要求 dx 也小
        if (E_x_now < relax_E_tol) and (np.max(np.abs(dx)) < relax_dx_tol):
            ok_count += 1
        else:
            ok_count = 0
    
        if ok_count >= n_consecutive_ok:
            relax_step_reached = k + 1
            relax_time_reached = t_now
            print(
                f"Reached fixed point by relaxation-energy criterion at step {k+1}, "
                f"time = {t_now:.6f}, E_x = {E_x_now:.6e}"
            )
            break

    n_hold = int(round(hold_time / p.dt))
    for k in range(n_hold):
        dx = -x + W @ phi(x, p)
        x = x + p.dt * dx
        t_now += p.dt

        if (k + 1) % store_every == 0:
            append_state_statistics(
                buffer=buffer,
                t=t_now,
                x=x,
                x_ref=x_ref,
                p=p,
                info=info,
                phi_ref=phi_ref,
            )

    # 保证最后时刻一定被记录
    if len(buffer["t"]) == 0 or (not np.isclose(buffer["t"][-1], t_now)):
        append_state_statistics(
            buffer=buffer,
            t=t_now,
            x=x,
            x_ref=x_ref,
            p=p,
            info=info,
            phi_ref=phi_ref,
        )

    stats = finalize_stats_buffer(buffer)

    # shift time: perturbation starts at t = 0
    stats["t"] = stats["t"] - stats["t"][-1]

    stats["x_end"] = x.copy()
    stats["relax_step_reached"] = relax_step_reached
    stats["relax_time_reached"] = relax_time_reached
    return stats


# ============================================================
# perturbation builder
# ============================================================

def build_perturbation_from_config(
    p: MultiAreaParams,
    rng: np.random.Generator,
    perturbation_cfg: dict,
) -> Tuple[np.ndarray, str]:
    """
    Build perturbation from a case config dictionary.

    Supported kinds:
    - "constant"
    - "gaussian"
    - "array"

    Both "constant" and "gaussian" accept an optional `area_weights` key
    (a list/tuple of per-area multipliers). For "gaussian", the optional
    `exact_l2_match_to` sub-config builds a reference perturbation and the
    Gaussian is then rescaled per-area so its area-wise L2 norms match
    the reference's area-wise L2 norms.
    """
    kind = perturbation_cfg["kind"]
    name = perturbation_cfg.get("name", kind)

    if kind == "constant":
        delta = constant_perturbation(
            p=p,
            amplitude=perturbation_cfg["amplitude"],
            area_weights=perturbation_cfg.get("area_weights", None),
        )
        return np.asarray(delta, dtype=float), name

    elif kind == "gaussian":
        exact_l2_match_to = None
        match_cfg = perturbation_cfg.get("exact_l2_match_to", None)

        if match_cfg is not None:
            exact_l2_match_to, _ = build_perturbation_from_config(
                p=p,
                rng=rng,
                perturbation_cfg=match_cfg,
            )

        delta = gaussian_perturbation(
            p=p,
            sigma=perturbation_cfg["sigma"],
            rng=rng,
            zero_mean=perturbation_cfg.get("zero_mean", True),
            area_weights=perturbation_cfg.get("area_weights", None),
            exact_l2_match_to=exact_l2_match_to,
        )
        return np.asarray(delta, dtype=float), name

    elif kind == "array":
        delta = np.asarray(perturbation_cfg["values"], dtype=float)
        info = population_info(p)
        if delta.shape != (info["N_total"],):
            raise ValueError(
                f"array perturbation must have shape ({info['N_total']},), got {delta.shape}"
            )
        return delta, name

    else:
        raise ValueError(f"Unknown perturbation kind: {kind}")


# ============================================================
# saving
# ============================================================

def save_case_results(
    output_dir: str | Path,
    case_name: str,
    p,
    case_cfg: dict,
    network_meta: dict,
    result: dict,
    eig_jac: np.ndarray,
    x_ref: np.ndarray,
    x_end: np.ndarray,
    perturbation: np.ndarray,
    scalar_summary: dict,
    trial_idx: int,
):
    output_dir = Path(output_dir)
    # case_dir = output_dir / sanitize_name(case_name)
    case_dir = output_dir / sanitize_name(case_name) / f"trial_{trial_idx:03d}"
    case_dir.mkdir(parents=True, exist_ok=True)

    # ---------- CSV ----------
    header = [
        "t",
        "is_post_perturbation",
        "E_x_total",
        "E_phi_total",
        "max_re_lambda_stability",
    ]

    columns = [
        result["t"],
        result["is_post_perturbation"].astype(float),
        result["E_x_total"],
        result["E_phi_total"],
        result["max_re_lambda_stability"],
    ]

    n_areas = result["E_x_by_area"].shape[1]
    for area_id in range(n_areas):
        header.append(f"E_x_area_{area_id}")
        columns.append(result["E_x_by_area"][:, area_id])

    for area_id in range(n_areas):
        header.append(f"E_phi_area_{area_id}")
        columns.append(result["E_phi_by_area"][:, area_id])

    csv_data = np.column_stack(columns)
    np.savetxt(
        case_dir / "timeseries.csv",
        csv_data,
        delimiter=",",
        header=",".join(header),
        comments="",
    )

    # ---------- NPZ ----------
    np.savez_compressed(
        case_dir / "timeseries.npz",
        t=result["t"],
        is_post_perturbation=result["is_post_perturbation"],
        E_x_total=result["E_x_total"],
        E_phi_total=result["E_phi_total"],
        E_x_by_area=result["E_x_by_area"],
        E_phi_by_area=result["E_phi_by_area"],
        max_re_lambda_stability=result["max_re_lambda_stability"],
        eig_jac=eig_jac,
        x_ref=x_ref,
        x_end=x_end,
        perturbation=perturbation,
    )

    # ---------- summary JSON ----------
    summary = {
        "case_name": case_name,
        #"network_meta": to_serializable(network_meta),
        "scalar_summary": to_serializable(scalar_summary),
        "case_config": to_serializable(case_cfg),
        "multi_area_params": to_serializable(p),
    }

    with open(case_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    return case_dir

In [7]:
def run_single_case(case_cfg: dict, output_dir: str | Path):
    case_name = case_cfg["case_name"]
    # p = case_cfg["p"]
    trial_idx = int(case_cfg.get("trial_idx", 0))
    trial_seed = int(case_cfg.get("trial_seed", case_cfg["p"].seed))
    p = replace(case_cfg["p"], seed=trial_seed)

    init_scale = case_cfg.get("init_scale", 0.5)
    relax_E_tol=case_cfg.get("relax_E_tol", 1e-6)
    relax_dx_tol=case_cfg.get("relax_dx_tol", 1e-2)
    max_relax_steps = case_cfg.get("max_relax_steps", 300000)
    hold_time = case_cfg.get("hold_time", 3.0)
    response_T = case_cfg.get("response_T", 30.0)
    store_every = case_cfg.get("store_every", 10)

    print("\n" + "=" * 70)
    print(f"Running case: {case_name}")

    # --------------------------------------------------------
    # base connectivity
    # --------------------------------------------------------
    W_base, meta = build_multi_area_connectivity(p)
    print("Network meta:")
    print("N_total =", meta["N_total"])
    print("Fraction of non-zero entries =", meta["nnz"] / (meta["N_total"] * meta["N_total"]))

    # --------------------------------------------------------
    # exact fixed point
    # --------------------------------------------------------
    x_star = solve_full_fixed_point(W=W_base, p=p)
    x_ref = x_star.copy()

    print("\nFull fixed point solved.")
    print("x_star mean =", np.mean(x_star))
    print("x_star max  =", np.max(x_star))
    print("x_star min  =", np.min(x_star))


    # --------------------------------------------------------
    # relaxation + hold (ONLY ONCE PER CASE)
    # initial point is sampled on a sphere around x_star
    # use a fully random seed here, independent of p.seed / case_cfg
    # --------------------------------------------------------
    rng_pre = np.random.default_rng()
    
    z = rng_pre.normal(loc=0.0, scale=1.0, size=meta["N_total"])
    
    # 与后续 mean perturbation 分开：去掉均匀方向分量
    z = z - np.mean(z)
    
    norm_z = np.linalg.norm(z)
    if norm_z == 0.0:
        raise RuntimeError("Random sphere sample has zero norm, please rerun.")
    
    delta_pre = init_scale * z / norm_z
    x0 = x_star + delta_pre
    
    pre_stats = relax_to_fixed_point_and_collect_statistics(
        W=W_base,
        p=p,
        x_init=x0,
        x_ref=x_ref,
        relax_E_tol=relax_E_tol,
        relax_dx_tol=relax_dx_tol,
        max_relax_steps=max_relax_steps,
        hold_time=hold_time,
        store_every=store_every,
    )
    
    x_end = pre_stats["x_end"].copy()
    
    residual_x_end = np.max(np.abs(-x_end + W_base @ phi(x_end, p)))
    residual_x_star = np.max(np.abs(-x_star + W_base @ phi(x_star, p)))
    dist_x_end_to_x_star = np.linalg.norm(x_end - x_star) / np.sqrt(len(x_star))
    
    print("||x_end - x_star||_2 / sqrt(N) =", dist_x_end_to_x_star)
    print("residual at x_end  =", residual_x_end)
    print("residual at x_star =", residual_x_star)
    
    # --------------------------------------------------------
    # Jacobian spectrum (ONLY ONCE PER CASE)
    # --------------------------------------------------------
    eig_jac = compute_stability_spectrum(
        W=W_base,
        p=p,
        x_star=x_ref,
    )
    max_re_lambda = float(np.max(eig_jac.real))
    print("max Re(lambda_stability) =", max_re_lambda)

    # --------------------------------------------------------
    # run all perturbations from the SAME x_end
    # --------------------------------------------------------
    rng_post = np.random.default_rng()
    perturbation_results = []

    for perturbation_cfg in case_cfg["perturbations"]:
        perturbation, perturbation_name = build_perturbation_from_config(
            p=p,
            rng=rng_post,
            perturbation_cfg=perturbation_cfg,
        )

        print("\n" + "-" * 50)
        print(f"Perturbation: {perturbation_name}")
        print("perturbation L2 norm =", np.linalg.norm(perturbation))

        post_out = simulate_dynamics(
            W=W_base,
            p=p,
            x_init=x_end + perturbation,
            T=response_T,
            store_every=store_every,
        )

        post_times = np.asarray(post_out["t"], dtype=float)
        post_traj = np.asarray(post_out["x"], dtype=float)

        post_stats = statistics_from_trajectory(
            times=post_times,
            traj=post_traj,
            x_ref=x_ref,
            p=p,
        )

        merged = merge_pre_post_statistics(
            pre_stats=pre_stats,
            post_stats=post_stats,
            max_re_lambda_stability=max_re_lambda,
        )

        scalar_summary = {
            "perturbation_name": perturbation_name,
            "trial_idx": int(trial_idx),
            "trial_seed": int(trial_seed),
            "max_re_lambda_stability": max_re_lambda,
            "x_end_vs_x_star_l2_per_sqrtN": float(dist_x_end_to_x_star),
            "residual_at_x_end": float(residual_x_end),
            "residual_at_x_star": float(residual_x_star),
            "perturbation_l2_norm": float(np.linalg.norm(perturbation)),
            "n_timepoints_total": int(merged["t"].shape[0]),
        }

        # case_dir = save_case_results(...)   # original (single-run path)
        trial_dir = save_case_results(
            output_dir=Path(output_dir) / sanitize_name(case_name),
            case_name=perturbation_name,
            p=p,
            case_cfg=case_cfg,
            network_meta=meta,
            result=merged,
            eig_jac=eig_jac,
            x_ref=x_ref,
            x_end=x_end,
            perturbation=perturbation,
            scalar_summary=scalar_summary,
            trial_idx=trial_idx,
        )

        print(f"Saved perturbation results to: {trial_dir}")

        perturbation_results.append({
            "perturbation_name": perturbation_name,
            "trial_idx": int(trial_idx),
            "trial_seed": int(trial_seed),
            "trial_dir": str(trial_dir),
            "summary": scalar_summary,
        })

    return {
        "case_name": case_name,
        "perturbation_results": perturbation_results,
    }


# ============================================================
# Wrapper: run n_trials independent realizations of the same case.
# Reads n_trials and trial_seed_base from case_cfg (defaults to a
# single trial). To change the number of trials, edit n_trials in
# build_predefined_cases / make_two_area_case_config.
# ============================================================

def run_case_with_trials(case_cfg: dict, output_dir: str | Path):
    case_name = case_cfg["case_name"]
    n_trials = int(case_cfg.get("n_trials", 1))
    trial_seed_base = int(case_cfg.get("trial_seed_base", 0))

    print("\n" + "#" * 70)
    print(f"Case: {case_name}  ->  n_trials = {n_trials}, seed_base = {trial_seed_base}")
    print("#" * 70)

    trials = []
    for k in range(n_trials):
        # inject per-trial knobs into a shallow copy so we do not mutate
        # the caller's case_cfg
        cfg_k = dict(case_cfg)
        cfg_k["trial_idx"] = k
        cfg_k["trial_seed"] = trial_seed_base + k

        out = run_single_case(cfg_k, output_dir)
        trials.append(out)

    return {
        "case_name": case_name,
        "n_trials": n_trials,
        "trials": trials,
    }


In [8]:
def make_two_area_case_config(
    *,
    case_name: str,
    area1: AreaParams,
    area2: AreaParams,
    s12: float = 0.0,
    s21: float = 0.0,
    J12: float = 0.0,
    J21: float = 0.0,
    activation: str = "threshold_linear",
    phi_max: float = np.inf,
    gamma: float = 0.5,
    dt: float = 0.005,
    T: float = 60.0,
    seed: int = 0,
    perturbations: list | None = None,
    init_scale: float = np.sqrt(4000),
    relax_E_tol: float = 1e-6,
    relax_dx_tol: float = 1e-2,
    max_relax_steps: int = 300000,
    hold_time: float = 0.0,
    response_T: float = 50.0,
    store_every: int = 10,
    rng_seed: int | None = None,
    n_trials: int = 10,
    trial_seed_base: int = 0,
):
    s_cross = np.zeros((2, 2), dtype=float)
    J_cross = np.zeros((2, 2), dtype=float)

    s_cross[0, 1] = s12
    s_cross[1, 0] = s21
    J_cross[0, 1] = J12
    J_cross[1, 0] = J21

    p = MultiAreaParams(
        areas=(area1, area2),
        s_cross=s_cross,
        J_cross=J_cross,
        allow_autapses=False,
        activation=activation,
        phi_max=phi_max,
        gamma=gamma,
        dt=dt,
        T=T,
        seed=seed,
    )

    if perturbations is None:
        perturbations = [
            {
                "kind": "constant",
                "name": "mean perturbation",
                "amplitude": 1.0,
                "area_weights": [1.0, 1.0],
            },
            {
                "kind": "gaussian",
                "name": "Gaussian perturbation",
                "sigma": 1.0,
                "zero_mean": True,
                "exact_l2_match_to": {
                    "kind": "constant",
                    "amplitude": 1.0,
                    "area_weights": [1.0, 1.0],
                },
            },
        ]

    return {
        "case_name": case_name,
        "p": p,
        "perturbations": perturbations,
        "init_scale": init_scale,
        "relax_E_tol": relax_E_tol,
        "relax_dx_tol": relax_dx_tol,
        "max_relax_steps": max_relax_steps,
        "hold_time": hold_time,
        "response_T": response_T,
        "store_every": store_every,
        "rng_seed": seed if rng_seed is None else rng_seed,
        "n_trials": n_trials,
        "trial_seed_base": trial_seed_base,
    }

In [9]:
def build_predefined_cases():
    N = 2000
    s_local = 0.05
    f_exc = 0.8
    g_inh = 4.5

    NE = int(N * f_exc)

    # --------------------------------------------------------
    # symmetric perturbations: both areas driven equally
    # --------------------------------------------------------
    mean_perturbation = {
        "kind": "constant",
        "name": "mean perturbation",
        "amplitude": 1.0,
        "area_weights": [1.0, 1.0],
    }

    gaussian_perturbation = {
        "kind": "gaussian",
        "name": "Gaussian perturbation",
        "sigma": 1.0,
        "zero_mean": True,
        "area_weights": [1.0, 1.0],
        "exact_l2_match_to": {
            "kind": "constant",
            "amplitude": 1.0,
            "area_weights": [1.0, 1.0],
        },
    }

    # --------------------------------------------------------
    # area-0-only perturbations: targeted stimulation of area 0,
    # area 1 starts at the fixed point and is driven only through
    # the cross-area connectivity
    # --------------------------------------------------------
    mean_perturbation_area0 = {
        "kind": "constant",
        "name": "mean perturbation, area 0 only",
        "amplitude": 1.0,
        "area_weights": [1.0, 0.0],
    }

    gaussian_perturbation_area0 = {
        "kind": "gaussian",
        "name": "Gaussian perturbation, area 0 only",
        "sigma": 1.0,
        "zero_mean": True,
        "area_weights": [1.0, 0.0],
        "exact_l2_match_to": {
            "kind": "constant",
            "amplitude": 1.0,
            "area_weights": [1.0, 0.0],
        },
    }

    perturbations_for_comparison = [
        # mean_perturbation,
        # gaussian_perturbation,
        mean_perturbation_area0,
        gaussian_perturbation_area0,
    ]

    cases = []

    cases.append(
        make_two_area_case_config(
            case_name="case_1_diskdom_005",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0430, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0430, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0108,
            J21=0.0108,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )

    cases.append(
        make_two_area_case_config(
            case_name="case_2_diskdom_010",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0408, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0408, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0102,
            J21=0.0102,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )

    cases.append(
        make_two_area_case_config(
            case_name="case_3_diskdom_020",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0362, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0362, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0091,
            J21=0.0091,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )

    cases.append(
        make_two_area_case_config(
            case_name="case_1_outlierdom_005",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0163, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0163, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0278,
            J21=0.0278,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )
        
    cases.append(
        make_two_area_case_config(
            case_name="case_2_outlierdom_010",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0165, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0165, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0266,
            J21=0.0266,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )

    cases.append(
        make_two_area_case_config(
            case_name="case_3_outlierdom_020",
            area1=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0168, g_inh=g_inh),
            area2=AreaParams(N=N, s=s_local, f_exc=f_exc, J_exc=0.0168, g_inh=g_inh),
            s12=0.025,
            s21=0.025,
            J12=0.0242,
            J21=0.0242,
            activation="threshold_linear",
            phi_max=np.inf,
            gamma=0.5,
            seed=0,
            perturbations=perturbations_for_comparison,
            response_T=50.0,
            store_every=10,
        )
    )

    return cases

In [10]:
def main():
    output_dir = "simulation_output_energy_10times_N2000"
    cases = build_predefined_cases()

    all_case_summaries = []

    print("=" * 80)
    print(f"Total number of cases to run: {len(cases)}")
    print("=" * 80)

    for i, case_cfg in enumerate(cases, start=1):
        print("\n" + "#" * 80)
        print(f"Start case {i}/{len(cases)}: {case_cfg['case_name']}")
        print("#" * 80)

        # out = run_single_case(case_cfg=case_cfg, output_dir=output_dir)
        out = run_case_with_trials(
            case_cfg=case_cfg,
            output_dir=output_dir,
        )
        all_case_summaries.append(out)

    print("\n" + "=" * 80)
    print("All predefined cases finished.")
    print("=" * 80)

    for item in all_case_summaries:
        print(f"\n{item['case_name']}:  ({item['n_trials']} trials)")
        # for sub in item["perturbation_results"]: ...   # original (flat list)
        for trial_out in item["trials"]:
            for sub in trial_out["perturbation_results"]:
                s = sub["summary"]
                print(
                    f"  - trial {sub['trial_idx']:03d} | {sub['perturbation_name']}: "
                    f"max Re(lambda_stability) = {s['max_re_lambda_stability']:.6e}, "
                    f"residual_at_x_end = {s['residual_at_x_end']:.6e}, "
                    f"saved in {sub['trial_dir']}"
                )


if __name__ == "__main__":
    main()

Total number of cases to run: 6

################################################################################
Start case 1/6: case_1_diskdom_005
################################################################################

######################################################################
Case: case_1_diskdom_005  ->  n_trials = 10, seed_base = 0
######################################################################

Running case: case_1_diskdom_005
Network meta:
N_total = 4000
Fraction of non-zero entries = 0.035

Full fixed point solved.
x_star mean = 0.0010010026907988356
x_star max  = 0.0010010026907988465
x_star min  = 0.0010010026907988246
Reached fixed point by relaxation-energy criterion at step 38881, time = 194.405000, E_x = 9.989303e-07
||x_end - x_star||_2 / sqrt(N) = 0.0009994649957303203
residual at x_end  = 0.00011805662393952858
residual at x_star = 9.99314585256074e-07
max Re(lambda_stability) = -0.02654248934064742

----------------------------------------